# Manhattan Distance — Ground Truth × Gherkin Generations (JSON)

Notebook adapted to the current data format:

- **Ground truth**: JSON with `cases`, where each case contains `case_id`, `original_case`, `reference_id`, and `gherkin`.
- **Generations**: JSON with the metadata `model`, `technique`, `number_of_executions` and, for each case, a `generations` list.
- The association between reference and generation is performed by **`case_id`**, not by the case position in the file.
- The **Manhattan Distance** calculation preserves the logic of the original notebook: each text pair is vectorized with `CountVectorizer` and compared using `cityblock`.
- The notebook accepts **1 ground truth file and 1 or more generation files** in the same upload.
- At the end, a **detailed CSV** containing all comparisons and rankings is generated.

> **Interpretation:** the lower the Manhattan Distance, the closer the term-count vectors of the two scenarios are under this representation.


In [ ]:
# ============================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================

import json
import re
import warnings

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import cityblock

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)

# If True, the CSV will be downloaded automatically at the end in Google Colab.
AUTO_DOWNLOAD_CSV = True


In [ ]:
# ============================================================
# 2. UPLOAD AND AUTOMATIC JSON IDENTIFICATION
# ============================================================

def load_json_bytes(file_name, content):
    # Accepts UTF-8 with or without BOM.
    try:
        text = content.decode("utf-8-sig")
        return json.loads(text)
    except Exception as error:
        raise ValueError(f"Could not read '{file_name}' as JSON: {error}") from error


def classify_json(file_name, data):
    # Classifies the file as ground_truth, geracoes, or desconhecido.
    # The original internal labels are preserved for compatibility.
    if not isinstance(data, dict):
        return "desconhecido"

    cases = data.get("cases")
    if not isinstance(cases, list):
        return "desconhecido"

    if data.get("is_reference_base") is True:
        return "ground_truth"

    if cases:
        first_case = cases[0]
        if isinstance(first_case, dict):
            if "generations" in first_case:
                return "geracoes"
            if "reference_id" in first_case and "gherkin" in first_case:
                return "ground_truth"

    return "desconhecido"


def process_upload(uploaded):
    files_data = []
    for file_name, content in uploaded.items():
        if not file_name.lower().endswith(".json"):
            print(f"⚠ Ignored (not JSON): {file_name}")
            continue

        data = load_json_bytes(file_name, content)
        file_type = classify_json(file_name, data)

        # These internal field names are preserved exactly as in the original notebook.
        files_data.append({"nome": file_name, "tipo": file_type, "dados": data})

    return files_data


try:
    from google.colab import files
except ImportError as error:
    raise RuntimeError(
        "This notebook was prepared for interactive upload in Google Colab. "
        "Run it in Colab or adapt this cell for local file reading."
    ) from error


# ------------------------------------------------------------
# STEP 1 — Ground truth
# ------------------------------------------------------------
print("STEP 1/2 — Upload the ground truth JSON file:")
ground_truth_upload = files.upload()
json_files = process_upload(ground_truth_upload)

ground_truths = [item for item in json_files if item["tipo"] == "ground_truth"]
generation_files = [item for item in json_files if item["tipo"] == "geracoes"]
unknown_files = [item["nome"] for item in json_files if item["tipo"] == "desconhecido"]

if unknown_files:
    print("⚠ JSON file(s) with unrecognized structure:", unknown_files)

if len(ground_truths) != 1:
    raise ValueError(
        f"Exactly 1 ground truth file is required. {len(ground_truths)} were identified. "
        "Check whether the file contains 'is_reference_base': true or cases with "
        "'reference_id' and 'gherkin'."
    )

ground_truth_file_name = ground_truths[0]["nome"]
ground_truth = ground_truths[0]["dados"]

print(f"\n✓ Ground truth identified: {ground_truth_file_name}")


# ------------------------------------------------------------
# STEP 2 — Generations
# ------------------------------------------------------------
# If generation files were uploaded together with the ground truth, reuse them.
# Otherwise, open a second file selector.
if not generation_files:
    print("\nSTEP 2/2 — Now upload one or more generation JSON files:")
    generation_upload = files.upload()
    new_files = process_upload(generation_upload)

    new_ground_truths = [
        item for item in new_files if item["tipo"] == "ground_truth"
    ]
    if new_ground_truths:
        print(
            "⚠ Additional ground truth ignored during the generation-file step:",
            [item["nome"] for item in new_ground_truths]
        )

    new_unknown_files = [
        item["nome"] for item in new_files if item["tipo"] == "desconhecido"
    ]
    if new_unknown_files:
        print("⚠ JSON file(s) with unrecognized structure:", new_unknown_files)

    generation_files.extend(
        item for item in new_files if item["tipo"] == "geracoes"
    )

if not generation_files:
    raise ValueError(
        "No generation file was identified. Generation files must contain "
        "'cases' and, within each case, the 'generations' key."
    )

print(f"\n✓ Generation files identified: {len(generation_files)}")
for generation_file in generation_files:
    data = generation_file["dados"]
    print(
        f"  - {generation_file['nome']} | model={data.get('model')} | "
        f"technique={data.get('technique')} | "
        f"declared executions={data.get('number_of_executions')}"
    )


In [ ]:
# ============================================================
# 3. DATA VALIDATION
# ============================================================

def index_ground_truth(ground_truth_data):
    indexed_references = {}
    duplicates = []

    for case in ground_truth_data.get("cases", []):
        case_id = case.get("case_id")
        if not case_id:
            continue

        if case_id in indexed_references:
            duplicates.append(case_id)

        indexed_references[case_id] = case

    if duplicates:
        raise ValueError(
            f"Duplicate case_id value(s) in the ground truth: {sorted(set(duplicates))}"
        )

    return indexed_references


references = index_ground_truth(ground_truth)
validation_warnings = []

for generation_file in generation_files:
    file_name = generation_file["nome"]
    data = generation_file["dados"]
    file_case_ids = []
    declared_executions = data.get("number_of_executions")

    for case in data.get("cases", []):
        case_id = case.get("case_id")
        file_case_ids.append(case_id)

        if case_id not in references:
            validation_warnings.append(
                f"{file_name}: {case_id} exists in the generations but not in the ground truth."
            )
            continue

        reference_original_case = references[case_id].get("original_case")
        generated_original_case = case.get("original_case")

        if (
            reference_original_case is not None
            and generated_original_case is not None
            and reference_original_case != generated_original_case
        ):
            validation_warnings.append(
                f"{file_name}: divergent original_case in {case_id}."
            )

        generations = case.get("generations", [])
        if (
            declared_executions is not None
            and len(generations) != declared_executions
        ):
            validation_warnings.append(
                f"{file_name}: {case_id} contains {len(generations)} generations, "
                f"but the file declares {declared_executions}."
            )

        executions = [
            generation.get("execution")
            for generation in generations
        ]
        valid_executions = [
            execution
            for execution in executions
            if execution is not None
        ]

        if len(valid_executions) != len(set(valid_executions)):
            validation_warnings.append(
                f"{file_name}: duplicate execution numbers in {case_id}."
            )

    reference_ids = set(references)
    generation_ids = set(file_case_ids)

    missing_case_ids = sorted(reference_ids - generation_ids)
    if missing_case_ids:
        validation_warnings.append(
            f"{file_name}: {len(missing_case_ids)} ground-truth case_id value(s) "
            f"do not appear in the generations. Examples: {missing_case_ids[:10]}"
        )

print(f"Cases in ground truth: {len(references)}")

if validation_warnings:
    print(f"\n⚠ {len(validation_warnings)} validation warning(s) were found:")
    for warning in validation_warnings:
        print(" -", warning)
else:
    print("\n✓ Structure validated without warnings.")


In [ ]:
# ============================================================
# 4. METRIC: MANHATTAN DISTANCE
# ============================================================

def manhattan_distance(reference_text, generated_text):
    # Preserves the exact strategy of the original notebook:
    # CountVectorizer on the text pair + cityblock (L1).
    reference_text = "" if reference_text is None else str(reference_text)
    generated_text = "" if generated_text is None else str(generated_text)

    if not reference_text.strip() and not generated_text.strip():
        return 0.0

    try:
        matrix = CountVectorizer().fit_transform(
            [reference_text, generated_text]
        ).toarray()
    except ValueError as error:
        warnings.warn(
            f"Could not vectorize a pair; distance set to NaN. Error: {error}"
        )
        return float("nan")

    return float(cityblock(matrix[0], matrix[1]))


In [ ]:
# ============================================================
# 5. COMPARISON CALCULATION
# ============================================================

results = []

for generation_file in generation_files:
    generations_file_name = generation_file["nome"]
    data = generation_file["dados"]

    model = data.get("model", "")
    technique = data.get("technique", "")
    declared_executions = data.get("number_of_executions")

    for generated_case in data.get("cases", []):
        case_id = generated_case.get("case_id")
        reference = references.get(case_id)

        if reference is None:
            continue

        reference_gherkin = reference.get("gherkin", "")

        for generation in generated_case.get("generations", []):
            generated_gherkin = generation.get("gherkin", "")
            distance_value = manhattan_distance(
                reference_gherkin,
                generated_gherkin
            )

            # Compatibility-critical output field names are preserved
            # exactly as in the original notebook.
            results.append({
                "arquivo_ground_truth": ground_truth_file_name,
                "arquivo_geracoes": generations_file_name,
                "modelo": model,
                "tecnica": technique,
                "execucoes_declaradas": declared_executions,
                "case_id": case_id,
                "source_id": reference.get("source_id"),
                "source_line": reference.get("source_line"),
                "original_case": reference.get("original_case"),
                "reference_id": reference.get("reference_id"),
                "generation_id": generation.get("generation_id"),
                "execucao": generation.get("execution"),
                "distancia_manhattan": distance_value,
                "gherkin_ground_truth": reference_gherkin,
                "gherkin_gerado": generated_gherkin,
            })

results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No comparison could be calculated.")

ranking_keys = ["arquivo_geracoes", "modelo", "tecnica", "case_id"]

results_df = results_df.sort_values(
    ranking_keys + ["distancia_manhattan", "execucao"],
    kind="stable",
    na_position="last"
).reset_index(drop=True)

# As in the original notebook, ranking is sequential after sorting
# by the lowest distance; ties continue to occupy successive positions.
results_df["ranking_no_caso"] = (
    results_df.groupby(ranking_keys, dropna=False).cumcount() + 1
)

columns = [
    "modelo",
    "tecnica",
    "case_id",
    "source_id",
    "original_case",
    "execucao",
    "distancia_manhattan",
    "ranking_no_caso",
    "generation_id",
    "reference_id",
    "gherkin_ground_truth",
    "gherkin_gerado",
    "execucoes_declaradas",
    "arquivo_ground_truth",
    "arquivo_geracoes",
]

results_df = results_df[columns]

print(f"✓ Comparisons calculated: {len(results_df):,}")
print(f"✓ Cases evaluated: {results_df['case_id'].nunique():,}")


In [ ]:
# ============================================================
# 6. ORGANIZED TABLES
# ============================================================

overall_summary_df = (
    results_df
    .groupby(["arquivo_geracoes", "modelo", "tecnica"], dropna=False)
    .agg(
        casos=("case_id", "nunique"),
        comparacoes=("distancia_manhattan", "count"),
        distancia_media=("distancia_manhattan", "mean"),
        distancia_mediana=("distancia_manhattan", "median"),
        desvio_padrao=("distancia_manhattan", "std"),
        distancia_minima=("distancia_manhattan", "min"),
        distancia_maxima=("distancia_manhattan", "max"),
    )
    .reset_index()
)

print("OVERALL SUMMARY")
display(
    overall_summary_df.style.format({
        "distancia_media": "{:.3f}",
        "distancia_mediana": "{:.3f}",
        "desvio_padrao": "{:.3f}",
        "distancia_minima": "{:.3f}",
        "distancia_maxima": "{:.3f}",
    })
)

case_summary_df = (
    results_df
    .groupby(
        ["arquivo_geracoes", "modelo", "tecnica", "case_id", "original_case"],
        dropna=False
    )
    .agg(
        execucoes_avaliadas=("execucao", "count"),
        distancia_media=("distancia_manhattan", "mean"),
        distancia_mediana=("distancia_manhattan", "median"),
        desvio_padrao=("distancia_manhattan", "std"),
        melhor_distancia=("distancia_manhattan", "min"),
        pior_distancia=("distancia_manhattan", "max"),
    )
    .reset_index()
    .sort_values(["modelo", "tecnica", "case_id"])
)

print("\nSUMMARY BY CASE — first 30 rows")
display(
    case_summary_df.head(30).style.format({
        "distancia_media": "{:.3f}",
        "distancia_mediana": "{:.3f}",
        "desvio_padrao": "{:.3f}",
        "melhor_distancia": "{:.3f}",
        "pior_distancia": "{:.3f}",
    })
)

print("\nDETAILED COMPARISONS — first 50 rows")
display_columns = [
    "modelo",
    "tecnica",
    "case_id",
    "original_case",
    "execucao",
    "distancia_manhattan",
    "ranking_no_caso",
]

display(
    results_df[display_columns]
    .head(50)
    .style
    .format({"distancia_manhattan": "{:.3f}"})
)


In [ ]:
# ============================================================
# 7. QUICK CASE LOOKUP
# ============================================================

def view_case(case_id):
    # Displays all executions of a case_id from the lowest to highest distance.
    case_results = results_df[
        results_df["case_id"] == case_id
    ].copy()

    if case_results.empty:
        print(f"No result found for {case_id}.")
        return

    columns = [
        "modelo",
        "tecnica",
        "case_id",
        "original_case",
        "execucao",
        "distancia_manhattan",
        "ranking_no_caso",
        "gherkin_ground_truth",
        "gherkin_gerado",
    ]

    display(
        case_results[columns]
        .sort_values(
            ["modelo", "tecnica", "distancia_manhattan", "execucao"]
        )
        .style
        .format({"distancia_manhattan": "{:.3f}"})
    )


first_case_id = results_df["case_id"].iloc[0]
print(f"Query example: {first_case_id}")
view_case(first_case_id)

# To query another case:
# view_case("TC_261")


In [ ]:
# ============================================================
# 8. CSV EXPORT
# ============================================================

def slug(text):
    text = str(text or "").strip().lower()
    text = re.sub(r"[^a-z0-9._-]+", "-", text)
    text = re.sub(r"-+", "-", text).strip("-")
    return text or "sem-identificacao"


unique_metadata = (
    results_df[["modelo", "tecnica"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

if len(unique_metadata) == 1:
    model = unique_metadata.loc[0, "modelo"]
    technique = unique_metadata.loc[0, "tecnica"]

    # The original output filename pattern is preserved for pipeline compatibility.
    csv_file_name = (
        f"metricas_manhattan_{slug(model)}_{slug(technique)}.csv"
    )
else:
    csv_file_name = "metricas_manhattan_multiplos_modelos_tecnicas.csv"

# UTF-8 with BOM helps Excel open accented field names correctly.
results_df.to_csv(
    csv_file_name,
    index=False,
    encoding="utf-8-sig"
)

print(f"✓ CSV generated: {csv_file_name}")
print(f"✓ Rows exported: {len(results_df):,}")

if AUTO_DOWNLOAD_CSV:
    try:
        from google.colab import files
        files.download(csv_file_name)
    except Exception as error:
        print(f"Automatic download was not completed: {error}")
